## Import Library

In [69]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, GridSearchCV, train_test_split
from sklearn.linear_model import LassoCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import make_scorer
from sklearn.feature_selection import SelectFromModel
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import dask.dataframe as dd
import lightgbm as lgb
import optuna
# import xgboost as xgb
# import catboost as cb
import seaborn as sns
from scipy.stats import uniform, randint

## Model Building and Data Handling

### Define scoring using myscore

In [70]:
def weighted_mae_calculaion(y_true, y_pred, weights):
    errors = np.abs(y_true - y_pred) * weights
    weighted_errors = errors*weights
    return np.sum(weighted_errors) / np.sum(weights)
def weighted_mae(y_true, y_pred, weights):
    y_true = y_true.merge(weights, on = 'unique_id', how = 'left')
    weights = y_true['weight'].values
    y_true = y_true['sales'].values
    return weighted_mae_calculaion(y_true, y_pred, weights)

### Get weight

In [71]:
# read from "test_weights.csv" using read.csv
weights = pd.read_csv("test_weights.csv")

## Model Training

### Import Data

In [72]:
file_path = "not_encoded_sales_train.csv"
test_file_path = "not_encoded_sales_test.csv"

### Train Model

In [74]:
chunk_size = 1000000  # Adjust based on your memory capacity
train_ratio = 0.8
total_rows = sum(1 for _ in open("not_encoded_sales_train.csv")) - 1
total_chunks = total_rows // chunk_size
train_chunks = int(total_chunks * train_ratio)

In [77]:
def objective(trial):
    try:
        # Hyperparameter space
        param = {
            "objective": "regression",
            "metric": "mae",
            "boosting_type": "gbdt",
            "n_estimators": trial.suggest_int("n_estimators", 500, 1500),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 31, 150),
            "max_depth": trial.suggest_int("max_depth", 6, 16),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 50, 400),
            "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.005, 0.2),
            "max_bin": trial.suggest_int("max_bin", 255, 512),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-4, 10, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-4, 10, log=True),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
            "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
            "verbosity": -1
        }

        model = None  # Initialize model as None before training
        valid_preds = []
        valid_y = []
        valid_unique_ids = []  # To keep track of unique_ids for validation
        train_data = None  # Initialize train dataset
        trained = False  # Track whether model has been trained

        for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size)):  
            # Convert categorical columns
            chunk['warehouse'] = chunk['warehouse'].astype('category')
            chunk['holiday_name'] = chunk['holiday_name'].astype('category')

            # Split features and target
            X_chunk = chunk.drop(columns=['sales'])
            y_chunk = chunk['sales']

            # Create dataset
            chunk_data = lgb.Dataset(
                X_chunk, label=y_chunk, 
                categorical_feature=['warehouse', 'holiday_name'], 
                free_raw_data=False
            )

            # Training phase
            if i < train_chunks:
                if model is None:  # First chunk initializes training
                    train_data = chunk_data
                    model = lgb.train(param, train_data)
                else:
                    model = lgb.train(
                        param, chunk_data, num_boost_round=100, 
                        init_model=model, keep_training_booster=True
                    )
                trained = True  # Mark that training has occurred

            # Validation phase
            else:
                if not trained:  # Ensure model is trained before predicting
                    print(f"Skipping validation because model wasn't trained.")
                    return float("inf")  # Stop this trial

                valid_preds.extend(model.predict(X_chunk))  # Extend instead of overwriting
                valid_y.extend(y_chunk)
                valid_unique_ids.extend(chunk['unique_id'])

        # Compute weighted MAE
        valid_df = pd.DataFrame({"unique_id": valid_unique_ids, "sales": valid_y})
        wmae = weighted_mae(valid_df, valid_preds, weights)

        return wmae
    
    except Exception as e:
        print(f"Trial {trial.number} failed due to error: {str(e)}")
        return float("inf")

In [78]:
# sales_train, sales_test = PCA_transformer(sales_train, sales_test)
# sales_train = lgb.Dataset(sales_train.drop(columns = ['sales']), sales_train['sales'])
# sales_train.save_binary("sales_train.bin")
# sales_train = lgb.Dataset("sales_train.bin")
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
study = optuna.create_study(direction="minimize", pruner=pruner)
study.optimize(objective, n_trials=2)

# Print best trial
print("Best trial: score {},\nparams {}".format(study.best_trial.value, study.best_trial.params))

[I 2025-02-13 21:34:15,319] A new study created in memory with name: no-name-8ae94eb5-ce20-457f-895c-4b35e53549ae
c:\Users\NINGZHI_JIANG\anaconda3\Lib\site-packages\lightgbm\engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")
c:\Users\NINGZHI_JIANG\anaconda3\Lib\site-packages\lightgbm\engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")
c:\Users\NINGZHI_JIANG\anaconda3\Lib\site-packages\lightgbm\engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")
[I 2025-02-13 21:38:48,595] Trial 0 finished with value: 657.27212636312 and parameters: {'n_estimators': 1079, 'learning_rate': 0.049076059575677775, 'num_leaves': 147, 'max_depth': 9, 'min_data_

Best trial: score 569.148991222508,
params {'n_estimators': 680, 'learning_rate': 0.034315431115833316, 'num_leaves': 43, 'max_depth': 8, 'min_data_in_leaf': 398, 'min_gain_to_split': 0.11994023103877673, 'max_bin': 322, 'lambda_l1': 3.1923862901353446, 'lambda_l2': 0.00518044429212551, 'feature_fraction': 0.763252661233738, 'bagging_fraction': 0.9848200717762652, 'bagging_freq': 6}


### Load the model and Make Prediction

In [79]:
best_params = study.best_params
model = None

for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size)):
    X_chunk = chunk.drop(columns = ['sales'])
    X_chunk['warehouse'] = X_chunk['warehouse'].astype('category')
    X_chunk['holiday_name'] = X_chunk['holiday_name'].astype('category')
    y_chunk = chunk['sales']
    train_data = lgb.Dataset(X_chunk, label = y_chunk, categorical_feature = ['warehouse', 'holiday_name'], free_raw_data = False)
    if i< train_chunks:
        if model is None:
            model = lgb.train(best_params, train_data)
        else:
             model = lgb.train(best_params, train_data, init_model = model, keep_training_booster = True)
model.save_model("final_lightgbm_model.txt")

c:\Users\NINGZHI_JIANG\anaconda3\Lib\site-packages\lightgbm\engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")
c:\Users\NINGZHI_JIANG\anaconda3\Lib\site-packages\lightgbm\engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")
c:\Users\NINGZHI_JIANG\anaconda3\Lib\site-packages\lightgbm\engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")


In [80]:
chunk_size = 10000
predictions = []
for chunk in pd.read_csv(test_file_path, chunksize=chunk_size):
    chunk[['warehouse', 'holiday_name']] = chunk[['warehouse', 'holiday_name']].astype('category')
    X_test = chunk
    preds = model.predict(X_test)
    predictions.extend(preds)
pred_df = pd.DataFrame({'predictions': predictions})
pred_df.to_csv("final_predictions.csv", index=False)